In [1]:
# ======================================================================
# DUBLIN PROPERTY PRICE DATA (ready to use)
# ======================================================================
# Input:  the national processed Property Price Register file
#         (dataSet/processed_dataset/property_price_dataset/ppr_2021_onwards.csv)
#
# Same steps as dublin_ber_data and dublin_planning_data:
#   1. load the property price CSV
#   2. keep only Dublin, give each sale an id, and add the postal district
#   3. remove the column that is the same thing as the target
#   4. remove columns with more than 50% missing values
#   5. remove constant columns
#   6. remove columns that point at one exact home (address, full Eircode)
#   7. put the columns in a clear order
#
# Targets (Student 3 + group model):
#   price_incl_vat         -> sale price in euro (regression, used as log price)
#   not_full_market_price  -> Yes / No (classification, Logistic regression)
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [2]:
# Folder paths

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

ppr_file = os.path.join(base, 'dataSet', 'processed_dataset', 'property_price_dataset', 'ppr_2021_onwards.csv')

print(ppr_file)

In [3]:
# Loading the property price file (one row per sale)

df_ppr = pd.read_csv(ppr_file, low_memory=False)

print('Rows:', df_ppr.shape[0], ' Columns:', df_ppr.shape[1])
df_ppr.head()

Rows: 352379  Columns: 12


,sale_date,address,county,eircode,price,not_full_market_price,vat_exclusive,property_description,property_size,price_incl_vat,year,quarter
0,2021-01-01,"APT 15, SANDON COVE, CASTLE AVE DUBLIN 3",Dublin,D03RY60,295000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,295000.0000,2021,2021Q1
1,2021-01-01,"APT 32, DANES COURT, CLONTARF RD DUBLIN 3",Dublin,D03W300,310000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,310000.0000,2021,2021Q1
2,2021-01-03,"16 BURLEIGH COURT, BURLINGTON ROAD, DUBLIN 4",Dublin,NaN,450000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,450000.0000,2021,2021Q1
3,2021-01-04,"1 CASTLEGAL CROSS, CLIFFONEY, CO. SLIGO",Sligo,NaN,148000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,148000.0000,2021,2021Q1
4,2021-01-04,"1 PLAZA VIEW, FAIR GREEN, MALLOW",Cork,NaN,158590.3,No,Yes,New Dwelling house /Apartment,NaN,179999.9905,2021,2021Q1


In [4]:
# Which counties are in the file?

df_ppr['county'].value_counts()

county
Dublin       106619
Cork          39526
Kildare       21210
Galway        15944
Meath         15441
Wicklow       13140
Wexford       12786
Limerick      12531
Louth         10618
Waterford     10027
Tipperary      9362
Donegal        8913
Kerry          8661
Mayo           8428
Clare          7642
Westmeath      6850
Laois          6594
Kilkenny       5912
Sligo          4999
Offaly         4888
Roscommon      4875
Cavan          4852
Carlow         3861
Leitrim        2958
Longford       2927
Monaghan       2815
Name: count, dtype: int64

In [5]:
# Keeping only the Dublin sales and saving them as df_dublin_ppr

df_dublin_ppr = df_ppr[df_ppr['county'] == 'Dublin'].copy()

print('All of Ireland:', len(df_ppr))
print('Dublin only:   ', len(df_dublin_ppr))

All of Ireland: 352379
Dublin only:    106619


In [6]:
# The register has no id column, so each sale gets its own number.
# (Like ber_id and ApplicationNumber: it is not a feature, it just tells us
#  which sale is which - e.g. to look up a sale that Isolation Forest flags.)

df_dublin_ppr.insert(0, 'sale_id', range(1, len(df_dublin_ppr) + 1))

df_dublin_ppr.head()

,sale_id,sale_date,address,county,eircode,price,not_full_market_price,vat_exclusive,property_description,property_size,price_incl_vat,year,quarter
0,1,2021-01-01,"APT 15, SANDON COVE, CASTLE AVE DUBLIN 3",Dublin,D03RY60,295000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,295000.000,2021,2021Q1
1,2,2021-01-01,"APT 32, DANES COURT, CLONTARF RD DUBLIN 3",Dublin,D03W300,310000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,310000.000,2021,2021Q1
2,3,2021-01-03,"16 BURLEIGH COURT, BURLINGTON ROAD, DUBLIN 4",Dublin,NaN,450000.0,No,No,Second-Hand Dwelling house /Apartment,NaN,450000.000,2021,2021Q1
9,4,2021-01-04,"12 Oileain Na Cranoige.Cranogue Isl, Balbutche...",Dublin,NaN,192951.0,No,Yes,New Dwelling house /Apartment,NaN,218999.385,2021,2021Q1
13,5,2021-01-04,"17 Belltree Lane, Beltree, Clonggriffin",Dublin,NaN,343612.0,No,Yes,New Dwelling house /Apartment,NaN,389999.620,2021,2021Q1


In [7]:
# Adding the postal district (same district names as the BER and planning data).
#   1st try: the Eircode       'D12X2Y3'                 -> 'Dublin 12'
#   2nd try: the address       '... CRUMLIN, DUBLIN 12'  -> 'Dublin 12'
# We do this now, before the address and Eircode columns are removed later.

dublin_districts = ['Dublin 1', 'Dublin 2', 'Dublin 3', 'Dublin 4', 'Dublin 5', 'Dublin 6',
                    'Dublin 6W', 'Dublin 7', 'Dublin 8', 'Dublin 9', 'Dublin 10', 'Dublin 11',
                    'Dublin 12', 'Dublin 13', 'Dublin 14', 'Dublin 15', 'Dublin 16', 'Dublin 17',
                    'Dublin 18', 'Dublin 20', 'Dublin 22', 'Dublin 24']

eircode = df_dublin_ppr['eircode'].fillna('').str.upper().str.strip()
address = df_dublin_ppr['address'].fillna('').str.upper()

# Eircode starting with D + two digits (or D6W)
eircode_number = eircode.str.extract(r'^D(\d\d|6W)')[0]

# 'DUBLIN' followed by a number (maybe with W, for 6W)
address_number = address.str.extract(r'DUBLIN\s*(\d+W?)')[0]

# use the Eircode first; if there is none, use the address
district_number = eircode_number.fillna(address_number)

# '04' -> '4' (Eircodes write a leading zero)
district_number = district_number.str.lstrip('0')

df_dublin_ppr['dublin_district'] = 'Dublin ' + district_number

In [8]:
# Anything that is not a real district becomes 'Dublin (no district)' -
# places like Swords (K67), Lucan (K78) or Blackrock (A94).

not_real = ~df_dublin_ppr['dublin_district'].isin(dublin_districts)
df_dublin_ppr.loc[not_real, 'dublin_district'] = 'Dublin (no district)'

has_district = (df_dublin_ppr['dublin_district'] != 'Dublin (no district)').mean() * 100
print('Sales with a postal district:', round(has_district, 1), '%')
df_dublin_ppr['dublin_district'].value_counts()

Sales with a postal district: 66.4 %


dublin_district
Dublin (no district)    35854
Dublin 15                7820
Dublin 24                5335
Dublin 18                5071
Dublin 8                 4350
Dublin 4                 4245
Dublin 7                 3778
Dublin 9                 3743
Dublin 13                3702
Dublin 12                3623
Dublin 3                 3307
Dublin 11                3290
Dublin 14                3203
Dublin 16                2981
Dublin 6                 2890
Dublin 22                2887
Dublin 5                 2797
Dublin 1                 2108
Dublin 6W                1734
Dublin 2                 1395
Dublin 10                 920
Dublin 17                 847
Dublin 20                 739
Name: count, dtype: int64

In [9]:
# Eircode area = the first 3 characters of the Eircode ('D12X2Y3' -> 'D12', 'K67...' -> 'K67').
# It is less exact than the full Eircode (which points at ONE home), so it is safe to keep,
# and it still tells apart places that have no district (Swords K67, Lucan K78 ...).

df_dublin_ppr['eircode_area'] = df_dublin_ppr['eircode'].str.upper().str.strip().str[:3]

print('Sales with an Eircode area:', round(df_dublin_ppr['eircode_area'].notnull().mean() * 100, 1), '%')
df_dublin_ppr['eircode_area'].value_counts().head(10)

Sales with an Eircode area: 78.8 %


eircode_area
D15    6970
D24    5070
D18    4488
D08    4050
D04    3900
A94    3683
A96    3644
D13    3563
D09    3493
D07    3468
Name: count, dtype: int64

In [10]:
# 6 sales describe the property in Irish:
#   'Teach/Árasán Cónaithe Atháimhe' = 'Second-Hand Dwelling house /Apartment'
# Using the English name so it is the same category.

df_dublin_ppr['property_description'] = df_dublin_ppr['property_description'].replace(
    'Teach/Árasán Cónaithe Atháimhe', 'Second-Hand Dwelling house /Apartment')

df_dublin_ppr['property_description'].value_counts()

property_description
Second-Hand Dwelling house /Apartment    87759
New Dwelling house /Apartment            18860
Name: count, dtype: int64

In [11]:
# Our price target is price_incl_vat.
# The register also has 'price': the price as recorded. For new homes it is
# recorded WITHOUT VAT, so price_incl_vat adds the VAT back. Let's check.

same_price = df_dublin_ppr['price'] == df_dublin_ppr['price_incl_vat']

print('Same value in both columns:', round(same_price.mean() * 100, 1), '%')
print()
print(pd.crosstab(df_dublin_ppr['vat_exclusive'], same_price))

Same value in both columns: 82.6 %

col_0          False  True 
vat_exclusive              
No                 0  88074
Yes            18545      0


In [12]:
# Removing the column that is the same thing as the target.
#   price -> it IS the sale price (identical for every second-hand home,
#            and only the VAT is different for new homes)
#
# We keep price_incl_vat and not_full_market_price (the targets).
# The price target must NEVER be used as an input for the price model.

same_as_target = ['price']

print('Columns before:', df_dublin_ppr.shape[1])
df_dublin_ppr = df_dublin_ppr.drop(columns=same_as_target)
print('Columns after: ', df_dublin_ppr.shape[1])

Columns before: 15
Columns after:  14


In [13]:
# Missing values in each column, as a percentage of all Dublin sales

missing_percent = df_dublin_ppr.isnull().sum() / len(df_dublin_ppr) * 100

missing_percent.round(2).sort_values(ascending=False)

property_size            100.00
eircode                   21.15
eircode_area              21.15
address                    0.00
sale_date                  0.00
sale_id                    0.00
not_full_market_price      0.00
county                     0.00
property_description       0.00
vat_exclusive              0.00
price_incl_vat             0.00
year                       0.00
quarter                    0.00
dublin_district            0.00
dtype: float64

In [14]:
# Removing the columns where more than 50% of the values are missing

columns_to_remove = missing_percent[missing_percent > 50].index

print('Columns removed:', len(columns_to_remove))
print(list(columns_to_remove))

print('Columns before:', df_dublin_ppr.shape[1])
df_dublin_ppr = df_dublin_ppr.drop(columns=columns_to_remove)
print('Columns after: ', df_dublin_ppr.shape[1])

Columns removed: 1
['property_size']
Columns before: 14
Columns after:  13


In [15]:
# Finding columns that have only ONE value for every sale.
# A column that never changes can't help the model, so we remove it.
# (nunique counts the different values, ignoring missing ones)

constant_columns = []

for column in df_dublin_ppr.columns:
    if df_dublin_ppr[column].nunique() <= 1:
        constant_columns.append(column)
        print(column, '->', df_dublin_ppr[column].unique()[:3])

print('Constant columns:', len(constant_columns))
print('Column names:', constant_columns)

county -> <StringArray>
['Dublin']
Length: 1, dtype: str


Constant columns: 1
Column names: ['county']


In [16]:
# Removing them

print('Columns before:', df_dublin_ppr.shape[1])
df_dublin_ppr = df_dublin_ppr.drop(columns=constant_columns)
print('Columns after: ', df_dublin_ppr.shape[1])

Columns before: 13
Columns after:  12


In [17]:
# Removing the columns that point at one exact home.
#   address -> already used to find dublin_district
#   eircode -> the full code is one exact home; we keep eircode_area instead
# They are not needed for modelling (data minimisation, as promised in our proposal).

exact_home_columns = ['address', 'eircode']

print('Columns before:', df_dublin_ppr.shape[1])
df_dublin_ppr = df_dublin_ppr.drop(columns=exact_home_columns)
print('Columns after: ', df_dublin_ppr.shape[1])

Columns before: 12
Columns after:  10


In [18]:
# Arranging the columns (nothing is removed - only the order changes):
# id -> dates -> categories -> targets

id_columns = ['sale_id']
date_columns = ['sale_date', 'year', 'quarter']
categorical_columns = ['dublin_district', 'eircode_area', 'property_description', 'vat_exclusive']
target_columns = ['price_incl_vat', 'not_full_market_price']

wanted_order = id_columns + date_columns + categorical_columns

# 1. columns from our lists, in our order
new_order = []
for column in wanted_order:
    if column in df_dublin_ppr.columns:
        new_order.append(column)

# 2. any column we did not list goes here, so it is NOT lost
for column in df_dublin_ppr.columns:
    if column not in new_order and column not in target_columns:
        new_order.append(column)
        print('Not in the lists, kept anyway:', column)

# 3. targets always at the end
for column in target_columns:
    if column in df_dublin_ppr.columns:
        new_order.append(column)

print('Columns before:', df_dublin_ppr.shape[1], ' after:', len(new_order))   # always the same
df_dublin_ppr = df_dublin_ppr[new_order]

df_dublin_ppr.head()

Columns before: 10  after: 10


,sale_id,sale_date,year,quarter,dublin_district,eircode_area,property_description,vat_exclusive,price_incl_vat,not_full_market_price
0,1,2021-01-01,2021,2021Q1,Dublin 3,D03,Second-Hand Dwelling house /Apartment,No,295000.000,No
1,2,2021-01-01,2021,2021Q1,Dublin 3,D03,Second-Hand Dwelling house /Apartment,No,310000.000,No
2,3,2021-01-03,2021,2021Q1,Dublin 4,NaN,Second-Hand Dwelling house /Apartment,No,450000.000,No
9,4,2021-01-04,2021,2021Q1,Dublin (no district),NaN,New Dwelling house /Apartment,Yes,218999.385,No
13,5,2021-01-04,2021,2021Q1,Dublin (no district),NaN,New Dwelling house /Apartment,Yes,389999.620,No


In [19]:
# Quick check of what we have now

print('Rows:', df_dublin_ppr.shape[0], ' Columns:', df_dublin_ppr.shape[1])
print(list(df_dublin_ppr.columns))

Rows: 106619  Columns: 10
['sale_id', 'sale_date', 'year', 'quarter', 'dublin_district', 'eircode_area', 'property_description', 'vat_exclusive', 'price_incl_vat', 'not_full_market_price']
